# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/real-huzaifa/flyrank-internship-ml-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The rule, in plain words

> A page goes to the top of my queue if its traffic is already falling inside March, and it
> still has enough traffic left to be worth saving. The faster it is falling and the more
> traffic it has, the higher it ranks.

### The score

score = decline_severity × volume_weight


- `decline_severity` = `max(0, −mar_h2_vs_h1)` — zero for anything flat or rising, up to 1.0 for
  a page that lost all its second-half traffic
- `volume_weight` = `log1p(mar_daily_impr)` — bigger pages count more, with log damping so one
  huge page cannot own the queue

I multiply rather than add so a page needs both. A collapsing page with 2 impressions a day
scores near zero, and so does a big stable page. My editor has 50 slots a week and both of those
would waste one.

### The reason codes

| reason code | condition | action label |
|---|---|---|
| `FALLING_HIGH_VOLUME` | momentum ≤ −0.20 and daily impressions ≥ 50 | `REFRESH_NOW` |
| `FALLING_LOW_VOLUME` | momentum ≤ −0.20 and daily impressions < 50 | `REVIEW_LATER` |
| `SOFT_DECLINE` | −0.20 < momentum ≤ −0.05 | `MONITOR` |
| `STABLE_OR_RISING` | momentum > −0.05 | `NO_ACTION` |

Checked in order, first match wins, so every row carries exactly one.

### Why this rule and not the obvious one

I checked two signals before writing it, and both came back negative. The tables are below.

**Signal 1 — worse position predicts decline: OPPOSITE.** Decline rate *falls* as position
worsens: 0.583 at ranks 1–3 down to 0.465 at 21–50. Median daily impressions drop from 50.5 to
12.0 across the same buckets, and a page getting 12 impressions a day cannot fall 20%. It is a
floor effect, not a content signal.

**Signal 2 — CTR below its position peers predicts decline: OPPOSITE, and barely measurable.**
Pages far above their peers decline least (0.395), pages below decline most (0.592) — the
reverse of the CTR-fix logic. And CTR is mostly absent rather than low: 45.2% of my pages had
zero clicks in March, rising to 92% at rank 50+. Checked directly, having any clicks at all is
worth 0.535 vs 0.479 — real but small.

Both are signals behind real FlyRank flags, and I am not building on either. That is the mistake
I found in the shipped `baseline_refresh_score`, which puts 30% of its weight on staleness while
page age correlates negatively with decline, and scores precision@50 = 0.340 against random's
0.542.

**What I use instead: within-month momentum.** `mar_h2_vs_h1` compares a page's second half of
March with its own first half. By quintile: 0.735 → 0.559 → 0.467 → 0.391 → 0.364, monotonic.
By decile the bottom is 0.797. I checked it is not a volume artifact — it holds inside all five
volume quintiles, 25 cells, no reversals — and volume amplifies it rather than explaining it.

Every input is measurable on 31 March. No position, no CTR, no `dim_content` field, no April.

In [4]:
# ── Signal checks: two bucket tables with n, before any rule ─────────────────
%pip -q install duckdb
import json
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

BASE    = "hf://datasets/FlyRank/internship-warehouse"
FACT_03 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_04 = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')"

# Same frame as notebook 03: features from March, label from April, no overlap.
frame = con.sql(f"""
WITH mar AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS mar_impr, SUM(gsc_clicks) AS mar_clicks,
         SUM(gsc_sum_position) AS mar_sum_pos, COUNT(*) AS mar_days,
         SUM(gsc_impressions) FILTER (report_date <  DATE '2026-03-16') AS h1_impr,
         SUM(gsc_impressions) FILTER (report_date >= DATE '2026-03-16') AS h2_impr
  FROM {FACT_03} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
),
apr AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS apr_impr, COUNT(*) AS apr_days
  FROM {FACT_04} WHERE gsc_data_available IS TRUE GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id, m.mar_impr, m.mar_clicks, m.mar_days,
       m.mar_impr    * 1.0 / m.mar_days             AS mar_daily_impr,
       m.mar_sum_pos * 1.0 / NULLIF(m.mar_impr, 0)  AS mar_avg_position,
       m.mar_clicks  * 100.0 / NULLIF(m.mar_impr, 0) AS mar_ctr,
       (m.h2_impr - m.h1_impr) * 1.0
         / NULLIF(m.h2_impr + m.h1_impr, 0)         AS mar_h2_vs_h1,
       a.apr_impr * 1.0 / NULLIF(a.apr_days, 0)     AS apr_daily_impr_raw
FROM mar m LEFT JOIN apr a USING (client_hash_id, content_hash_id)
WHERE m.mar_days >= 15 AND m.mar_impr >= 30
""").df()

frame["mar_h2_vs_h1"]   = frame["mar_h2_vs_h1"].fillna(0.0)
frame["apr_daily_impr"] = frame["apr_daily_impr_raw"].fillna(0.0)
frame["y"] = (frame["apr_daily_impr"] < 0.80 * frame["mar_daily_impr"]).astype(int)
frame = frame.drop(columns=["apr_daily_impr_raw"])

BASE_RATE = frame["y"].mean()
print(f"frame: {len(frame):,} pages | {frame['client_hash_id'].nunique()} clients "
      f"| base rate {BASE_RATE:.3f}\n")

# ── SIGNAL 1 — position (behind FlyRank's CTR-fix flag) ──────────────────────
frame["pos_bucket"] = pd.cut(frame["mar_avg_position"], bins=[0, 3, 10, 20, 50, 200],
                             labels=["1-3", "4-10", "11-20", "21-50", "50+"])
print("SIGNAL 1 — does WORSE position predict decline?")
print(frame.groupby("pos_bucket", observed=True).agg(
        n=("y", "size"), decline_rate=("y", "mean"),
        median_daily_impr=("mar_daily_impr", "median"),
        median_ctr=("mar_ctr", "median")).round(3).to_string())
print("VERDICT: OPPOSITE — decline rate FALLS as position worsens (.583 -> .465).")
print("         Low-traffic pages cannot drop 20%. This is a floor effect, not a signal.\n")

# ── SIGNAL 2 — CTR relative to position peers (the CTR-fix logic itself) ─────
frame["ctr_vs_peers"] = frame["mar_ctr"] - frame.groupby("pos_bucket", observed=True)["mar_ctr"].transform("median")
frame["ctr_gap_bucket"] = pd.cut(frame["ctr_vs_peers"], bins=[-100, -0.20, -0.02, 0.02, 0.20, 100],
                                 labels=["far below", "below", "at peers", "above", "far above"])
print("SIGNAL 2 — does CTR BELOW ITS POSITION PEERS predict decline?")
print(frame.groupby("ctr_gap_bucket", observed=True).agg(
        n=("y", "size"), decline_rate=("y", "mean"),
        median_ctr=("mar_ctr", "median"),
        median_position=("mar_avg_position", "median")).round(3).to_string())
print(f"   pages with ZERO clicks in March: {(frame['mar_ctr']==0).mean()*100:.1f}%")
print(frame.groupby("pos_bucket", observed=True).agg(
        n=("y", "size"), pct_zero_ctr=("mar_ctr", lambda s: (s == 0).mean() * 100)).round(1).to_string())
print("VERDICT: OPPOSITE — 'far above peers' declines LEAST (.395), 'below' most (.592).")
print("         And CTR is mostly absent, not low: 45.2% zero overall, 92% at rank 50+.\n")

# ── The signal I will actually use, and its confound check ───────────────────
frame["mom_q"] = pd.qcut(frame["mar_h2_vs_h1"], 5, labels=["Q1 falling", "Q2", "Q3", "Q4", "Q5 rising"])
frame["vol_q"] = pd.qcut(frame["mar_daily_impr"], 5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"])
print("WITHIN-MONTH MOMENTUM — the signal my rule uses")
print(frame.groupby("mom_q", observed=True).agg(
        n=("y", "size"), decline_rate=("y", "mean")).round(3).to_string())
print("\nCONFOUND CHECK — momentum inside each volume quintile (all 25 cells):")
print(frame.groupby(["vol_q", "mom_q"], observed=True)["y"].mean().unstack().round(3).to_string())
print("VERDICT: CONFIRMED — monotonic in every volume stratum. Not a volume artifact.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame: 116,539 pages | 40 clients | base rate 0.504

SIGNAL 1 — does WORSE position predict decline?
                n  decline_rate  median_daily_impr  median_ctr
pos_bucket                                                    
1-3         10355         0.583             50.500       0.188
4-10        52736         0.511             27.452       0.157
11-20       22546         0.501             14.806       0.028
21-50       23988         0.465             12.000       0.000
50+          6913         0.480              4.621       0.000
VERDICT: OPPOSITE — decline rate FALLS as position worsens (.583 -> .465).
         Low-traffic pages cannot drop 20%. This is a floor effect, not a signal.

SIGNAL 2 — does CTR BELOW ITS POSITION PEERS predict decline?
                    n  decline_rate  median_ctr  median_position
ctr_gap_bucket                                                  
below           41069         0.592       0.000            6.967
at peers        24509         0.484       0

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**How I evaluate it.** Precision@K on the same slice and the same April label my Week-5 model
will use, with the base rate printed next to it. Precision@50 of 0.80 means nothing until you
know random picking gives 0.504. Since K=50 is a small sample I also print the 5th–95th
percentile of 500 random draws — if my rule does not clear the top of that band, it is not doing
anything a coin could not.

**No thresholds are tuned against the metric.** All three cutoffs (−0.20, −0.05, 50 impressions
per day) come from the bucket tables in section 1 and were fixed before I scored anything.
Tuning them against precision@50 would make the baseline look good and mean nothing, and this
baseline's job is to be honestly beatable.

The CSV carries the rank, the two hash IDs, the score, the reason code, the action, and the
March inputs an editor would want to see. No April column goes into it.
nothing — and this baseline's job is to be honestly beatable.

In [5]:
# ── THE RULE: score, one reason code, one action label ───────────────────────
FALL_HARD, FALL_SOFT, VOL_FLOOR = -0.20, -0.05, 50.0   # all set from section 1's tables

frame["decline_severity"] = np.maximum(0.0, -frame["mar_h2_vs_h1"])
frame["volume_weight"]    = np.log1p(frame["mar_daily_impr"])
frame["baseline_score"]   = frame["decline_severity"] * frame["volume_weight"]

def classify(r):
    if r["mar_h2_vs_h1"] <= FALL_HARD and r["mar_daily_impr"] >= VOL_FLOOR:
        return "FALLING_HIGH_VOLUME", "REFRESH_NOW"
    if r["mar_h2_vs_h1"] <= FALL_HARD:
        return "FALLING_LOW_VOLUME",  "REVIEW_LATER"
    if r["mar_h2_vs_h1"] <= FALL_SOFT:
        return "SOFT_DECLINE",        "MONITOR"
    return "STABLE_OR_RISING",        "NO_ACTION"

frame[["reason_code", "action"]] = frame.apply(classify, axis=1, result_type="expand")

assert frame["reason_code"].notna().all(), "every row must carry exactly one reason code"
print("REASON CODES — one per row, first match wins")
print(frame.groupby(["reason_code", "action"], observed=True).agg(
        n=("y", "size"), decline_rate=("y", "mean"),
        median_momentum=("mar_h2_vs_h1", "median"),
        median_daily_impr=("mar_daily_impr", "median")).round(3).to_string())

# ── EVALUATE: precision@K against the base rate ──────────────────────────────
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = frame["y"].to_numpy()
rng = np.random.default_rng(42)
rand50 = np.array([y[rng.choice(len(y), 50, replace=False)].mean() for _ in range(500)])

print(f"\nbase rate (random picking): {BASE_RATE:.3f}")
print(f"{'K':>6} {'my rule':>10} {'lift':>8}")
print("-" * 26)
results = {}
for K in (10, 20, 50, 100, 500):
    p = precision_at_k(frame["baseline_score"], y, K)
    results[f"precision_at_{K}"] = round(float(p), 4)
    print(f"{K:>6} {p:>10.3f} {p - BASE_RATE:>+8.3f}")
print("-" * 26)
print(f"random@50: mean {rand50.mean():.3f}, 5th-95th pct {np.percentile(rand50,5):.3f}-{np.percentile(rand50,95):.3f}")

# ── WRITE THE QUEUE ──────────────────────────────────────────────────────────
Path("work/outputs").mkdir(parents=True, exist_ok=True)
queue = (frame.sort_values("baseline_score", ascending=False)
              .assign(rank=lambda d: range(1, len(d) + 1))
              [["rank", "client_hash_id", "content_hash_id", "baseline_score",
                "reason_code", "action", "mar_h2_vs_h1", "mar_daily_impr",
                "mar_avg_position", "mar_ctr"]])
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"\nwrote work/outputs/baseline_action_score.csv — {len(queue):,} rows, ranked")

metrics = {
    "as_of_date": "2026-03-31", "feature_window": "2026-03", "label_window": "2026-04",
    "n_pages": int(len(frame)), "n_clients": int(frame["client_hash_id"].nunique()),
    "base_rate": round(float(BASE_RATE), 4),
    "random_at_50_mean": round(float(rand50.mean()), 4),
    "rule": "score = max(0, -mar_h2_vs_h1) * log1p(mar_daily_impr)",
    "thresholds": {"fall_hard": FALL_HARD, "fall_soft": FALL_SOFT, "volume_floor": VOL_FLOOR},
    **results,
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/baseline_metrics.json")

REASON CODES — one per row, first match wins
                                      n  decline_rate  median_momentum  median_daily_impr
reason_code         action                                                               
FALLING_HIGH_VOLUME REFRESH_NOW    4788         0.800           -0.317            116.339
FALLING_LOW_VOLUME  REVIEW_LATER  15528         0.732           -0.362              8.056
SOFT_DECLINE        MONITOR       17952         0.597           -0.116             21.194
STABLE_OR_RISING    NO_ACTION     78271         0.420            0.178             20.516

base rate (random picking): 0.504
     K    my rule     lift
--------------------------
    10      0.900   +0.396
    20      0.900   +0.396
    50      0.880   +0.376
   100      0.820   +0.316
   500      0.858   +0.354
--------------------------
random@50: mean 0.501, 5th-95th pct 0.380-0.600

wrote work/outputs/baseline_action_score.csv — 116,539 rows, ranked
wrote work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**The confidence note that applies to all twenty:** they are the extreme tail of a single
signal. My score multiplies severity by volume, so the top of the queue is pages that fell hard
*and* still carry real traffic. I trust the *set* more than the *order* within it: scores
compress quickly, so rank 3 and rank 18 are closer than their positions suggest.

**What would make these wrong — four ways, in the order they worry me.**

1. **A publish date inside March.** A page that went live on 10 March has a first half that is
   partly pre-launch, so its collapse is an artifact of where I cut the month. I cannot check
   this: `dim_content` has no as-of snapshot, so I have no trustworthy publish date. This is my
   largest unquantified risk.
2. **Seasonal or event traffic.** A page about a March event should fall in April. Refreshing it
   wastes the slot and it recovers on its own next year.
3. **A tracking break.** If GSC stopped recording for one client mid-month, all its pages
   collapse together and none needs an editor. The client-concentration check below tests for
   this.
4. **Reversion.** Some drops are noise around a page's own average and come back without help.
   My rule cannot tell "still collapsing" from "already bottomed out" using March alone.

In [6]:
# ── TOP-20 REVIEW ────────────────────────────────────────────────────────────
top20 = queue.head(20).merge(
    frame[["client_hash_id", "content_hash_id", "y", "apr_daily_impr"]],
    on=["client_hash_id", "content_hash_id"], how="left")

review = pd.DataFrame({
    "rank":        top20["rank"],
    "client":      top20["client_hash_id"].str[-6:],
    "content":     top20["content_hash_id"].str[-6:],
    "score":       top20["baseline_score"].round(2),
    "action":      top20["action"],
    "reason":      top20["reason_code"],
    "momentum":    top20["mar_h2_vs_h1"].round(3),
    "mar_daily":   top20["mar_daily_impr"].round(1),
    "apr_daily":   top20["apr_daily_impr"].round(1),
    "declined":    np.where(top20["y"] == 1, "YES", "no"),
})
print("TOP 20 — the queue an editor would actually work down")
print(review.to_string(index=False))

hit = top20["y"].mean()
print(f"\nprecision@20 = {hit:.3f}   (base rate {BASE_RATE:.3f}, lift {hit - BASE_RATE:+.3f})")
print(f"misses in the top 20: {int((top20['y'] == 0).sum())}")

print("\nIS MY TOP 20 JUST ONE CLIENT? (a tracking break would look exactly like this)")
print(top20["client_hash_id"].str[-6:].value_counts().to_string())
print(f"   distinct clients in top 20: {top20['client_hash_id'].nunique()}")

print("\nHOW FAR DOWN DOES THE SCORE ACTUALLY SEPARATE?")
print(f"   rank 1 score  : {queue['baseline_score'].iloc[0]:.3f}")
print(f"   rank 20 score : {queue['baseline_score'].iloc[19]:.3f}")
print(f"   rank 50 score : {queue['baseline_score'].iloc[49]:.3f}")
print(f"   rank 500 score: {queue['baseline_score'].iloc[499]:.3f}")

TOP 20 — the queue an editor would actually work down
 rank client content  score      action              reason  momentum  mar_daily  apr_daily declined
    1 f265ea  0a3abb   7.89 REFRESH_NOW FALLING_HIGH_VOLUME    -0.999     2704.3        6.5      YES
    2 5d81d4  d97f04   5.74 REFRESH_NOW FALLING_HIGH_VOLUME    -0.988      332.0        1.0      YES
    3 9f63c4  ea94ca   5.65 REFRESH_NOW FALLING_HIGH_VOLUME    -0.840      829.2       65.2      YES
    4 5d81d4  9e83fd   5.52 REFRESH_NOW FALLING_HIGH_VOLUME    -0.998      251.2        0.0      YES
    5 5e0096  c01691   5.50 REFRESH_NOW FALLING_HIGH_VOLUME    -0.950      326.2        6.6      YES
    6 9f63c4  3fca87   5.47 REFRESH_NOW FALLING_HIGH_VOLUME    -0.715     2094.7      774.7      YES
    7 4ef01b  c31c17   5.36 REFRESH_NOW FALLING_HIGH_VOLUME    -0.711     1861.9     1990.3       no
    8 5d81d4  03face   5.31 REFRESH_NOW FALLING_HIGH_VOLUME    -0.989      213.9        1.0      YES
    9 f265ea  c01b7a   5.23 REFRESH_N

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Which picks look wrong

**The reason-code monoculture.** Every row in my top 20 carries the same code. Because the score
multiplies severity by volume, only the extreme of both can reach the top, so an editor working
this queue sees one story and gets no variety of evidence. A rule that surfaced a second reason
code up there would be more useful even at the same precision.

**The misses I cannot flag in advance.** The pages in the top 20 that did not decline in April
are the reversion cases: they fell in March and then held. Using March data alone I have no way
to separate them from the ones that kept falling, so they are not a bug I can fix — they are the
rule's ceiling.

**The ranking is thinner than it looks.** Scores compress steeply, so pages at rank 30 and rank
300 sit close together in score but far apart in the queue. Being in the top 50 is meaningful;
the exact order inside it is not, and an editor should not treat it as one.

**The one I cannot check at all.** A page published mid-March would look identical to a
collapsing page in my first-half/second-half comparison. No as-of snapshot exists, so I cannot
rule this out for any row.

### Leakage check

Three confirmations, asserted in the cell below rather than claimed here:

1. **No future window.** The score rebuilds exactly from March-only columns. April appears only
   as the label and in the review table, where I use it to score the rule after the fact. No
   April column is written to the CSV.
2. **No product flag.** No `trend_direction`, no `is_declining_label`, no
   `baseline_refresh_score`. The shipped rule is something I compare against, not something I
   feed in.
3. **No `dim_content` field.** Its dates post-date my decision moment, which is also why I
   cannot check the publish-date risk above.

### What Week 5 has to beat

Precision@50 from section 2, on this slice, against this April label, with the base rate beside
it. I am freezing those thresholds now — moving them once model work starts would convince
nobody, including me.

In [7]:
# ── LEAKAGE CHECK: prove the score uses March only ───────────────────────────
MARCH_ONLY = ["mar_impr", "mar_clicks", "mar_days", "mar_daily_impr",
              "mar_avg_position", "mar_ctr", "mar_h2_vs_h1"]
APRIL_COLS = ["apr_daily_impr", "y"]

rebuilt = np.maximum(0.0, -frame["mar_h2_vs_h1"]) * np.log1p(frame["mar_daily_impr"])
assert np.allclose(rebuilt, frame["baseline_score"]), "score does not rebuild from March columns"
print("[1] score rebuilds EXACTLY from March-only columns          PASS")
print(f"    inputs: {MARCH_ONLY}")

corr = {c: abs(np.corrcoef(frame[c], frame["y"])[0, 1]) for c in MARCH_ONLY}
print("\n[2] |correlation with the April label| — no March feature should be near 1.0")
for c, v in sorted(corr.items(), key=lambda kv: -kv[1]):
    print(f"    {c:20s} {v:.4f}")
assert max(corr.values()) < 0.90, "a March feature is suspiciously close to the label"
print("    max |r| below 0.90                                       PASS")

leaked_in_csv = [c for c in queue.columns if c in APRIL_COLS]
print(f"\n[3] April columns written to the queue CSV: {leaked_in_csv or 'none'}")
assert not leaked_in_csv, "an April column leaked into the deliverable"
print("    the CSV an editor receives contains no future data          PASS")

print("\n[4] no FlyRank product flag used:")
print("    trend_direction / is_declining_label / baseline_refresh_score — none present")
print(f"    columns in the scored frame: {sorted(frame.columns.tolist())}")

print(f"\nFROZEN BASELINE — precision@50 = {results['precision_at_50']:.3f} "
      f"vs base rate {BASE_RATE:.3f}. This is the number Week 5 must beat.")

[1] score rebuilds EXACTLY from March-only columns          PASS
    inputs: ['mar_impr', 'mar_clicks', 'mar_days', 'mar_daily_impr', 'mar_avg_position', 'mar_ctr', 'mar_h2_vs_h1']

[2] |correlation with the April label| — no March feature should be near 1.0
    mar_h2_vs_h1         0.2521
    mar_ctr              0.0982
    mar_clicks           0.0738
    mar_days             0.0475
    mar_daily_impr       0.0409
    mar_impr             0.0393
    mar_avg_position     0.0369
    max |r| below 0.90                                       PASS

[3] April columns written to the queue CSV: none
    the CSV an editor receives contains no future data          PASS

[4] no FlyRank product flag used:
    trend_direction / is_declining_label / baseline_refresh_score — none present
    columns in the scored frame: ['action', 'apr_daily_impr', 'baseline_score', 'client_hash_id', 'content_hash_id', 'ctr_gap_bucket', 'ctr_vs_peers', 'decline_severity', 'mar_avg_position', 'mar_clicks', 'mar_ctr', 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.